# 常見設計模式

## 學習目標

- 認識三大類設計模式：創建型、結構型、行為型
- 掌握 7 個最實用的模式，每個都能說出「解決什麼問題」
- 學會根據問題選擇適合的模式

---

## 創建型模式（Creational）

### Singleton：確保只有一個實例

**問題：** 某些物件（資料庫連線、設定檔）全程式只該有一份，建立多份會浪費資源或造成不一致。

**解法：** 覆寫 `__new__`，第一次建立時存起來，之後都回傳同一個。

In [ ]:
class Database:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.connection = "已連線"
        return cls._instance

db1 = Database()
db2 = Database()
print(db1 is db2)          # True，同一個物件
print(db1.connection)      # "已連線"

> 注意：Singleton 在多執行緒環境需要額外加鎖。Python 中更常見的做法是直接用模組層級變數。

---

### Factory：讓建立邏輯集中管理

**問題：** 根據不同條件建立不同類別的物件，散落在程式各處難以維護。

**解法：** 用一個工廠函式（或類別）統一處理建立邏輯。

In [ ]:
class Dog:
    def speak(self):
        return "汪汪！"

class Cat:
    def speak(self):
        return "喵喵！"

class Fish:
    def speak(self):
        return "......"

def animal_factory(animal_type):
    animals = {
        "dog": Dog,
        "cat": Cat,
        "fish": Fish,
    }
    cls = animals.get(animal_type)
    if cls is None:
        raise ValueError(f"未知動物類型: {animal_type}")
    return cls()

pet = animal_factory("cat")
print(pet.speak())  # "喵喵！"

新增動物時只要加一行到字典裡，不用改其他程式碼。

---

## 結構型模式（Structural）

### Adapter：讓不相容的介面合作

**問題：** 你有一個舊系統（或第三方套件），它的介面跟你的程式不相容，但你不能改它的原始碼。

**解法：** 寫一個轉接器，把舊介面轉成你需要的介面。

In [ ]:
# 舊系統：用 XML 輸出
class OldSystem:
    def get_xml_data(self):
        return "<data><name>小明</name></data>"

# 你的程式期望 dict
class OldSystemAdapter:
    def __init__(self, old_system):
        self._old = old_system

    def get_data(self):
        xml = self._old.get_xml_data()
        # 簡化示意，實際用 xml.etree 解析
        return {"name": "小明"}

adapter = OldSystemAdapter(OldSystem())
print(adapter.get_data())  # {"name": "小明"}

---

### Facade：簡化複雜子系統

**問題：** 子系統有很多類別和步驟，使用者每次都要記住正確的呼叫順序。

**解法：** 提供一個「門面」類別，把複雜流程包成簡單方法。

In [ ]:
class CPU:
    def start(self):
        print("CPU 啟動")

class Memory:
    def load(self):
        print("記憶體載入")

class Disk:
    def read(self):
        print("磁碟讀取")

# Facade：使用者只需要知道這個
class Computer:
    def __init__(self):
        self._cpu = CPU()
        self._memory = Memory()
        self._disk = Disk()

    def boot(self):
        """一鍵開機，隱藏複雜細節"""
        self._cpu.start()
        self._memory.load()
        self._disk.read()
        print("系統就緒！")

pc = Computer()
pc.boot()
# CPU 啟動 → 記憶體載入 → 磁碟讀取 → 系統就緒！

---

## 行為型模式（Behavioral）

### Observer：一對多的事件通知

**問題：** 一個物件狀態改變時，需要通知多個相關物件，但不想讓它們緊密耦合。

**解法：** 被觀察者維護一份觀察者清單，狀態改變時逐一通知。

In [ ]:
class WeatherStation:
    def __init__(self):
        self._observers = []
        self._temperature = 0

    def add_observer(self, observer):
        self._observers.append(observer)

    def set_temperature(self, temp):
        self._temperature = temp
        for observer in self._observers:
            observer.update(temp)

class PhoneDisplay:
    def update(self, temp):
        print(f"手機顯示：目前 {temp}°C")

class WindowDisplay:
    def update(self, temp):
        print(f"窗戶面板：目前 {temp}°C")

station = WeatherStation()
station.add_observer(PhoneDisplay())
station.add_observer(WindowDisplay())
station.set_temperature(25)
# 手機顯示：目前 25°C
# 窗戶面板：目前 25°C

---

### Strategy：動態切換演算法

**問題：** 同一件事有多種做法（排序、付款、壓縮），你想在執行時期切換，而不是用一堆 `if/elif`。

**解法：** 把每種做法封裝成獨立類別，透過組合注入。

In [ ]:
class CreditCardPayment:
    def pay(self, amount):
        print(f"信用卡付款 ${amount}")

class LinePayPayment:
    def pay(self, amount):
        print(f"LINE Pay 付款 ${amount}")

class CashPayment:
    def pay(self, amount):
        print(f"現金付款 ${amount}")

class ShoppingCart:
    def __init__(self, payment_strategy):
        self._payment = payment_strategy

    def checkout(self, amount):
        self._payment.pay(amount)

# 執行時期選擇付款方式
cart = ShoppingCart(LinePayPayment())
cart.checkout(500)  # "LINE Pay 付款 $500"

# 換一種方式
cart = ShoppingCart(CashPayment())
cart.checkout(500)  # "現金付款 $500"

---

### Template Method：固定流程，彈性步驟

**問題：** 多個類別的流程骨架相同，但某些步驟的實作不同。

**解法：** 父類別定義流程順序，子類別覆寫特定步驟。

In [ ]:
from abc import ABC, abstractmethod

class DataProcessor(ABC):
    def process(self):
        """流程骨架（不可覆寫）"""
        data = self.read_data()
        result = self.transform(data)
        self.save(result)

    @abstractmethod
    def read_data(self):
        pass

    @abstractmethod
    def transform(self, data):
        pass

    def save(self, result):
        print(f"儲存結果：{result}")

class CSVProcessor(DataProcessor):
    def read_data(self):
        return "csv,raw,data"

    def transform(self, data):
        return data.upper()

class JSONProcessor(DataProcessor):
    def read_data(self):
        return '{"key": "value"}'

    def transform(self, data):
        return data.replace("value", "processed")

CSVProcessor().process()
# 儲存結果：CSV,RAW,DATA

---

## 模式選擇指南

| 你遇到什麼問題？ | 用什麼模式 | 核心概念 |
|------------------|-----------|----------|
| 全程式只該有一份物件 | **Singleton** | 控制實例數量 |
| 建立邏輯散落各處、難以維護 | **Factory** | 集中建立邏輯 |
| 第三方介面不相容 | **Adapter** | 轉接介面 |
| 子系統太複雜，使用者難上手 | **Facade** | 簡化入口 |
| 狀態改變要通知多個物件 | **Observer** | 事件訂閱 |
| 同一行為有多種做法要切換 | **Strategy** | 演算法替換 |
| 流程相同但步驟不同 | **Template Method** | 骨架固定，細節彈性 |

### 快速判斷流程

```
需要控制物件建立？
  ├─ 只要一個 → Singleton
  └─ 依條件建不同物件 → Factory

需要處理介面問題？
  ├─ 舊介面不合用 → Adapter
  └─ 太多介面要簡化 → Facade

需要管理物件互動？
  ├─ 一對多通知 → Observer
  ├─ 多種演算法切換 → Strategy
  └─ 固定流程、彈性步驟 → Template Method
```

---

## 總結

設計模式不是必須套用的規則，而是前人解決常見問題的經驗總結。學習重點：

1. **先理解問題**，再找模式，不要為了用模式而用模式
2. **Python 有自己的慣用法**，例如用模組變數取代 Singleton、用函式取代簡單的 Strategy
3. **從小處開始**，遇到具體問題時再回來查這張表

---

上一篇：[迭代器與生成器](02_iterator_and_generator.ipynb)